# S9b.1 — Alembic Migration Setup

Validates: Alembic initialization, async env.py, initial migration, Makefile targets, model registration.

In [ ]:
import sys
sys.path.insert(0, "../..")

from pathlib import Path
PROJECT_ROOT = Path("../..")
print(f"Project root: {PROJECT_ROOT.resolve()}")

## 1. Verify alembic.ini

In [ ]:
from alembic.config import Config as AlembicConfig

ini_path = PROJECT_ROOT / "alembic.ini"
assert ini_path.exists(), "alembic.ini must exist"

config = AlembicConfig(str(ini_path))
assert config.get_main_option("script_location") == "alembic"
print("\n\u2713 alembic.ini exists and loads correctly")

## 2. Verify Alembic directory structure

In [ ]:
alembic_dir = PROJECT_ROOT / "alembic"
assert alembic_dir.is_dir()
assert (alembic_dir / "env.py").exists()
assert (alembic_dir / "versions").is_dir()
assert (alembic_dir / "script.py.mako").exists()
print("\n\u2713 Alembic directory structure is correct")

## 3. Verify async env.py

In [ ]:
env_content = (alembic_dir / "env.py").read_text()
assert "Base" in env_content, "Must import Base"
assert "target_metadata" in env_content, "Must set target_metadata"
assert "async" in env_content, "Must support async"
assert "get_settings" in env_content, "Must read config from settings"
assert "src.models" in env_content, "Must import models"
assert "paperalchemy_secret" not in env_content, "No hardcoded passwords"
print("\n\u2713 env.py is correctly configured for async + config")

## 4. Verify initial migration

In [ ]:
versions_dir = alembic_dir / "versions"
migrations = [f for f in versions_dir.glob("*.py") if not f.name.startswith("__")]
assert len(migrations) >= 1, "At least one migration must exist"

first = sorted(migrations)[0]
content = first.read_text()
assert "papers" in content
assert "def upgrade" in content
assert "def downgrade" in content
for col in ["arxiv_id", "title", "abstract", "authors", "published_date"]:
    assert col in content, f"Missing column: {col}"
print(f"\n\u2713 Initial migration found: {first.name}")
print(f"  Creates papers table with all expected columns")

## 5. Verify Makefile targets

In [ ]:
import re

makefile = (PROJECT_ROOT / "Makefile").read_text()
assert re.search(r"^db-migrate:", makefile, re.MULTILINE)
assert re.search(r"^db-upgrade:", makefile, re.MULTILINE)
assert re.search(r"^db-downgrade:", makefile, re.MULTILINE)
assert "--autogenerate" in makefile
assert "upgrade head" in makefile
assert "downgrade -1" in makefile
print("\n\u2713 Makefile has db-migrate, db-upgrade, db-downgrade targets")

## 6. Verify model registration

In [ ]:
from src.models import Paper
from src.db.base import Base

assert Paper.__tablename__ == "papers"
assert "papers" in Base.metadata.tables
print(f"\n\u2713 Paper model registered in Base.metadata")
print(f"  Tables: {list(Base.metadata.tables.keys())}")

## Summary

All S9b.1 checks passed:
- alembic.ini configured with `script_location = alembic`
- Async env.py reads DB URL from config, imports models
- Initial migration creates papers table with all columns
- Makefile targets: db-migrate, db-upgrade, db-downgrade
- Paper model registered in Base.metadata for autogenerate